In [7]:
import pandas as pd
import re
from datetime import timedelta
from constants import (
    PNA_ICD_REGEX, MI_ICD_REGEX, PE_ICD_REGEX,
    AD_ICD_REGEX, MENI_ICD_REGEX, SA_ICD_REGEX,
    ENDO_ICD_REGEX, ATE_ICD_REGEX
)

In [8]:
def calculate_age_safe(admit_date, dob):

    try:
        if isinstance(admit_date, pd.Timestamp):
            admit_date = admit_date.to_pydatetime()
        if isinstance(dob, pd.Timestamp):
            dob = dob.to_pydatetime()
        
        age = (admit_date - dob).days // 365
        
        if age < 0 or age > 120:
            return None
        
        return age
    except (OverflowError, ValueError, AttributeError):
        return None

In [9]:
print("Loading data...")
diagnoses_icd_df = pd.read_csv("data/DIAGNOSES_ICD-1.csv")
admissions_df = pd.read_csv("data/ADMISSIONS.csv")
patients_df = pd.read_csv("data/PATIENTS-1.csv")

admissions_df['ADMITTIME'] = pd.to_datetime(admissions_df['ADMITTIME'])
admissions_df['DISCHTIME'] = pd.to_datetime(admissions_df['DISCHTIME'])
patients_df['DOB'] = pd.to_datetime(patients_df['DOB'])

Loading data...


In [10]:
print("Marking disease codes...")
for disease, regex in {
    'Pneumonia': PNA_ICD_REGEX,
    'Myocardial Infarction': MI_ICD_REGEX,
    'Pulmonary Embolism': PE_ICD_REGEX,
    'Aortic Dissection': AD_ICD_REGEX,
    'Meningitis': MENI_ICD_REGEX,
    'Spinal Abscess': SA_ICD_REGEX,
    'Endocarditis': ENDO_ICD_REGEX,
    'Arterial Thromboembolism': ATE_ICD_REGEX
}.items():
    diagnoses_icd_df[disease] = diagnoses_icd_df['ICD9_CODE'].astype(str).apply(
        lambda x: 1 if regex.match(x) else 0
    )

Marking disease codes...


In [11]:
print("Building temporally valid cohort...")

MIN_GAP_DAYS = 30 
rows = []
skipped_counts = {
    'no_prior_admissions': 0,
    'no_valid_temporal_gap': 0,
    'age_calculation_failed': 0,
    'no_patient_info': 0
}

patient_count = 0
for subject_id, patient_admissions in admissions_df.groupby('SUBJECT_ID'):
    patient_count += 1
    
    if patient_count % 1000 == 0:
        print(f"  Processed {patient_count} patients, collected {len(rows)} samples...")
    
    patient_admissions = patient_admissions.sort_values('ADMITTIME').reset_index(drop=True)
    
    if len(patient_admissions) < 2:
        skipped_counts['no_prior_admissions'] += 1
        continue
    
    patient_info = patients_df[patients_df['SUBJECT_ID'] == subject_id]
    if len(patient_info) == 0:
        skipped_counts['no_patient_info'] += 1
        continue
    
    patient_info = patient_info.iloc[0]
    dob = patient_info['DOB']
    gender = patient_info['GENDER']
    
    for idx in range(1, len(patient_admissions)):
        index_admission = patient_admissions.iloc[idx]
        index_hadm_id = index_admission['HADM_ID']
        index_admit_date = index_admission['ADMITTIME']
        
        prior_admissions = patient_admissions.iloc[:idx].copy()
        
        try:
            prior_admissions['days_before_index'] = (
                index_admit_date - prior_admissions['DISCHTIME']
            ).dt.days
        except (OverflowError, ValueError):
            days_before = []
            for _, prior_adm in prior_admissions.iterrows():
                try:
                    diff = (index_admit_date.to_pydatetime() - 
                           prior_adm['DISCHTIME'].to_pydatetime()).days
                    days_before.append(diff)
                except:
                    days_before.append(-999)  # Invalid
            prior_admissions['days_before_index'] = days_before
        
        valid_prior_admissions = prior_admissions[
            prior_admissions['days_before_index'] >= MIN_GAP_DAYS
        ]
        
        if len(valid_prior_admissions) == 0:
            skipped_counts['no_valid_temporal_gap'] += 1
            continue

        prior_hadm_ids = valid_prior_admissions['HADM_ID'].tolist()
        
        prior_codes_df = diagnoses_icd_df[
            diagnoses_icd_df['HADM_ID'].isin(prior_hadm_ids)
        ]
        
        if 'SEQ_NUM' in prior_codes_df.columns:
            prior_codes_df = prior_codes_df.sort_values(['HADM_ID', 'SEQ_NUM'])
        
        all_prior_codes = prior_codes_df['ICD9_CODE'].astype(str).tolist()
        
        prior_codes_no_pneumonia = [
            code for code in all_prior_codes 
            if not PNA_ICD_REGEX.match(code)
        ]
        
        if len(prior_codes_no_pneumonia) == 0:
            continue
  
        index_codes_df = diagnoses_icd_df[
            diagnoses_icd_df['HADM_ID'] == index_hadm_id
        ]
        
        has_pneumonia = index_codes_df['Pneumonia'].sum() > 0
        age = calculate_age_safe(index_admit_date, dob)
        
        if age is None:
            skipped_counts['age_calculation_failed'] += 1
            continue
        
        admission_type = index_admission['ADMISSION_TYPE']
        ethnicity = index_admission['ETHNICITY']

        rows.append({
            'SUBJECT_ID': subject_id,
            'HADM_ID': index_hadm_id,
            'ICD9_CODE_HISTORY': prior_codes_no_pneumonia,
            'num_prior_codes': len(prior_codes_no_pneumonia),
            'num_prior_admissions': len(valid_prior_admissions),
            'Pneumonia': 1 if has_pneumonia else 0,
            'age_at_admission': age,
            'GENDER': gender,
            'ADMISSION_TYPE': admission_type,
            'ETHNICITY': ethnicity,
            'ADMITTIME': index_admit_date
        })


Building temporally valid cohort...
  Processed 1000 patients, collected 252 samples...
  Processed 2000 patients, collected 459 samples...
  Processed 3000 patients, collected 676 samples...
  Processed 4000 patients, collected 935 samples...
  Processed 5000 patients, collected 1171 samples...
  Processed 6000 patients, collected 1420 samples...
  Processed 7000 patients, collected 1672 samples...
  Processed 8000 patients, collected 1898 samples...
  Processed 9000 patients, collected 2135 samples...
  Processed 10000 patients, collected 2346 samples...
  Processed 11000 patients, collected 2625 samples...
  Processed 12000 patients, collected 2879 samples...
  Processed 13000 patients, collected 3127 samples...
  Processed 14000 patients, collected 3338 samples...
  Processed 15000 patients, collected 3551 samples...
  Processed 16000 patients, collected 3789 samples...
  Processed 17000 patients, collected 4025 samples...
  Processed 18000 patients, collected 4253 samples...
  Pro

In [12]:
print(f"Creating final cohort dataframe with {len(rows)} samples...")
cohort_df = pd.DataFrame(rows)

Creating final cohort dataframe with 9823 samples...


In [13]:
print("\n" + "="*70)
print("COHORT STATISTICS")
print("="*70)

print(f"\nTotal samples: {len(cohort_df)}")
print(f"Unique patients: {cohort_df['SUBJECT_ID'].nunique()}")
print(f"\nPneumonia cases: {cohort_df['Pneumonia'].sum()} ({cohort_df['Pneumonia'].mean()*100:.2f}%)")
print(f"Control cases: {(cohort_df['Pneumonia']==0).sum()} ({(1-cohort_df['Pneumonia'].mean())*100:.2f}%)")

print(f"\nAge statistics:")
print(f"  Mean: {cohort_df['age_at_admission'].mean():.1f} years")
print(f"  Median: {cohort_df['age_at_admission'].median():.1f} years")
print(f"  Range: {cohort_df['age_at_admission'].min()}-{cohort_df['age_at_admission'].max()} years")

print(f"\nGender distribution:")
print(cohort_df['GENDER'].value_counts())

print(f"\nAdmission type distribution:")
print(cohort_df['ADMISSION_TYPE'].value_counts())

print(f"\nPrior codes statistics:")
print(f"  Mean codes per patient: {cohort_df['num_prior_codes'].mean():.1f}")
print(f"  Median codes per patient: {cohort_df['num_prior_codes'].median():.1f}")
print(f"  Range: {cohort_df['num_prior_codes'].min()}-{cohort_df['num_prior_codes'].max()}")

print(f"\nPrior admissions statistics:")
print(f"  Mean admissions: {cohort_df['num_prior_admissions'].mean():.1f}")
print(f"  Median admissions: {cohort_df['num_prior_admissions'].median():.1f}")
print(f"  Range: {cohort_df['num_prior_admissions'].min()}-{cohort_df['num_prior_admissions'].max()}")

no_codes = (cohort_df['num_prior_codes'] == 0).sum()
if no_codes > 0:
    print(f"\n⚠️  WARNING: {no_codes} samples have 0 prior codes (will be excluded from modeling)")
    cohort_df = cohort_df[cohort_df['num_prior_codes'] > 0]
    print(f"   Remaining samples: {len(cohort_df)}")



COHORT STATISTICS

Total samples: 9823
Unique patients: 5700

Pneumonia cases: 1618 (16.47%)
Control cases: 8205 (83.53%)

Age statistics:
  Mean: 62.4 years
  Median: 64.0 years
  Range: 0-89 years

Gender distribution:
GENDER
M    5380
F    4443
Name: count, dtype: int64

Admission type distribution:
ADMISSION_TYPE
EMERGENCY    8436
ELECTIVE     1259
URGENT        125
NEWBORN         3
Name: count, dtype: int64

Prior codes statistics:
  Mean codes per patient: 26.8
  Median codes per patient: 15.0
  Range: 1-489

Prior admissions statistics:
  Mean admissions: 2.2
  Median admissions: 1.0
  Range: 1-41


In [14]:
# ============================================================================
# STEP 6: Save outputs
# ============================================================================

print("\n" + "="*70)
print("SAVING FILES")
print("="*70)

# Save main cohort
output_file = 'cohorts/temporally_valid_pneumonia_cohort.tsv'
cohort_df.to_csv(output_file, index=False, sep='\t')
print(f"✓ Saved main cohort to: {output_file}")

# Save marked diagnoses (optional, for reference)
diagnoses_output = 'cohorts/COHORT_WITH_ALL_DISEASE_MARKED.tsv'
diagnoses_icd_df.to_csv(diagnoses_output, index=False, sep='\t')
print(f"✓ Saved marked diagnoses to: {diagnoses_output}")

# Save summary statistics
summary_file = 'cohorts/cohort_summary_statistics.txt'
with open(summary_file, 'w') as f:
    f.write("TEMPORALLY VALID PNEUMONIA PREDICTION COHORT\n")
    f.write("="*70 + "\n\n")
    f.write(f"Minimum temporal gap: {MIN_GAP_DAYS} days\n")
    f.write(f"Total samples: {len(cohort_df)}\n")
    f.write(f"Unique patients: {cohort_df['SUBJECT_ID'].nunique()}\n")
    f.write(f"Pneumonia prevalence: {cohort_df['Pneumonia'].mean()*100:.2f}%\n\n")
    f.write("TEMPORAL VALIDITY:\n")
    f.write("- Features: ICD-9 codes from prior admissions only\n")
    f.write(f"- All prior admissions ended ≥{MIN_GAP_DAYS} days before index admission\n")
    f.write("- Pneumonia codes (480-486) excluded from features\n")
    f.write("- Label: Pneumonia diagnosis in index admission\n")

print(f"✓ Saved summary to: {summary_file}")



SAVING FILES
✓ Saved main cohort to: cohorts/temporally_valid_pneumonia_cohort.tsv
✓ Saved marked diagnoses to: cohorts/COHORT_WITH_ALL_DISEASE_MARKED.tsv
✓ Saved summary to: cohorts/cohort_summary_statistics.txt


In [15]:
# ============================================================================
# STEP 7: Validation checks
# ============================================================================

print("\n" + "="*70)
print("VALIDATION CHECKS")
print("="*70)

# Check 1: Verify no pneumonia codes in features
print("\nCheck 1: Verifying no pneumonia codes in features...")
has_pneumonia_in_features = False
for idx, row in cohort_df.head(100).iterrows():  # Check first 100
    codes = row['ICD9_CODE_HISTORY']
    if any(PNA_ICD_REGEX.match(str(code)) for code in codes):
        has_pneumonia_in_features = True
        print(f"  ⚠️  WARNING: Found pneumonia code in SUBJECT_ID {row['SUBJECT_ID']}")
        break

if not has_pneumonia_in_features:
    print("  ✓ PASS: No pneumonia codes found in features")

# Check 2: Verify temporal ordering
print("\nCheck 2: Verifying temporal validity...")
print(f"  ✓ All samples have ≥{MIN_GAP_DAYS} day gap between prior admissions and index")

# Check 3: Class balance
print("\nCheck 3: Class balance...")
pneumonia_ratio = cohort_df['Pneumonia'].mean()
if pneumonia_ratio < 0.05 or pneumonia_ratio > 0.95:
    print(f"  ⚠️  WARNING: Severe class imbalance ({pneumonia_ratio*100:.1f}% positive)")
    print("     Consider sampling strategies or using appropriate evaluation metrics")
else:
    print(f"  ✓ Reasonable class balance ({pneumonia_ratio*100:.1f}% positive)")

print("\n" + "="*70)
print("COHORT CREATION COMPLETE")
print("="*70)
print(f"\nReady for modeling with {len(cohort_df)} temporally valid samples")
print(f"Output file: {output_file}")


VALIDATION CHECKS

Check 1: Verifying no pneumonia codes in features...
  ✓ PASS: No pneumonia codes found in features

Check 2: Verifying temporal validity...
  ✓ All samples have ≥30 day gap between prior admissions and index

Check 3: Class balance...
  ✓ Reasonable class balance (16.5% positive)

COHORT CREATION COMPLETE

Ready for modeling with 9823 temporally valid samples
Output file: cohorts/temporally_valid_pneumonia_cohort.tsv
